# Update existing Salesforce Job__c

PATCH an existing `Job__c` record with fields from Supabase `job_current`.

- Set **`SUPABASE_JOB_ID`** and **`RECORD_ID`** below, then run all cells.
- `DRY_RUN = True` previews the payload without writing.

**Automated pipeline sync** (`sf_scrape_sync`): uses the same full `prepare_payload_for_write` field set as production scrapes. Test-only Salesforce fields (`test_status__c`, `test_posted_date__c`) are included only when **`PROXI_SF_TEST_MODE=true`** in `.env`.

## 1. Setup (path + `.env`)

Run first.

In [ ]:
import os
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
for _ in range(15):
    if (project_root / "src" / "utils").is_dir():
        break
    project_root = project_root.parent
else:
    raise RuntimeError("Could not locate repo root containing src/utils")

sys.path.insert(0, str(project_root / "src"))

from dotenv import load_dotenv

load_dotenv(project_root / ".env")

print("Project root:", project_root)
print("DB_PASSWORD set:", bool(os.environ.get("DB_PASSWORD") or os.environ.get("DATABASE_URL") or os.environ.get("DIRECT_URL")))
print("SALESFORCE_CONSUMER_KEY set:", bool(os.environ.get("SALESFORCE_CONSUMER_KEY")))

Project root: /Users/andylee/Desktop/projects/proxi/proxi_salesforce_automation
DB_PASSWORD set: True
SALESFORCE_CONSUMER_KEY set: True


## 2. Configuration

In [ ]:
import os

# --- Switch these for each run ---
SUPABASE_JOB_ID = "19596"  # job_current.job_id
RECORD_ID = "a015f00000cGr3EAAS"  # Salesforce Job__c Id (keep testing on this record)
SUPABASE_SCHEMA = "public"

DATA_SOURCE = "supabase"  # "manual" = use JOB_ROW dict pasted in §3

JOB_OBJECT = os.environ.get("SALESFORCE_JOB_OBJECT", "Job__c").strip()

USE_CANONICAL_DESCRIPTION = True
DESCR_USE_HTML = True  # None = use env PROXI_JOB_DESCRIPTION_HTML

HAND_FIELDS = {
    # "Kimedics_Job_ID__c": SUPABASE_JOB_ID,
}

DRY_RUN = False

## 3. Load `JOB_ROW`

**Supabase:** fetches one row, applies **account / worksite enrichment** (`job_sf_enrichment`).

**Manual:** set `DATA_SOURCE = "manual"` in §2 and define **`JOB_ROW`** below (must match `job_current` column names).

In [ ]:
if DATA_SOURCE == "supabase":
    from utils.supabase_db import load_job_current_row_for_salesforce

    JOB_ROW = load_job_current_row_for_salesforce(SUPABASE_JOB_ID, schema=SUPABASE_SCHEMA)
    print(
        "Loaded job_current:",
        "job_id=", JOB_ROW.get("job_id"),
        "city=", JOB_ROW.get("city"),
        "state=", JOB_ROW.get("state"),
        "updated_at=", JOB_ROW.get("updated_at"),
    )
else:
    JOB_ROW = {
        "job_id": "manual",
        "city": "",
        "state": "",
        "description_full_text": "",
    }
    print("Using manual JOB_ROW — edit the dict in this cell.")

Loaded job_current: job_id= 19587 city= Escanaba state= MI updated_at= 2026-04-08 19:32:29.655539+00:00


## 4. (Optional) Preview canonical description only

Skip if you only care about the full PATCH payload in §7.

In [ ]:
from utils.job_description_proxi_template import build_proxi_job_posting_description

_use = True if DESCR_USE_HTML is None else DESCR_USE_HTML
_preview_desc = build_proxi_job_posting_description(JOB_ROW, use_html=_use)
print(_preview_desc[:2000])
if len(_preview_desc) > 2000:
    print(f"\n… ({len(_preview_desc)} chars total)")

<p><strong>General Dentist Locum Tenens Opportunity in Escanaba, MI</strong></p><p><br/></p><p>We are seeking a General Dentist for a locum tenens opportunity in Escanaba, Michigan. This position offers the opportunity to practice comprehensive general dentistry with a supportive clinical team and steady patient flow.</p><p><br/></p><p>This role is ideal for a dentist comfortable with surgical extractions and dentures who enjoys working in a collaborative environment.</p><p><br/></p><p>Travel and lodging may be available for qualified candidates.</p><p><br/></p><p><strong>Dates:</strong> April 7-9<br/><strong>Schedule:</strong> 8a-5p</p><p><br/></p><p><strong>Clinical Scope</strong></p><ul style="margin-top:0;margin-bottom:0;padding-left:20px;"><li style="margin:0 0 4px 0;">Surgical extractions for dentures</li><li style="margin:0 0 4px 0;">extractions could include simple/ surgical/ full mouth- please notate any limitations in presentation</li></ul><p><br/></p><p>Full mouth extraction

## 5. Authenticate (Salesforce)

In [ ]:
import os

from utils.salesforce import SalesforceLoginError, get_token_auto

ck = os.environ.get("SALESFORCE_CONSUMER_KEY", "")
cs = os.environ.get("SALESFORCE_CONSUMER_SECRET", "")
if not ck or not cs:
    raise RuntimeError("Set SALESFORCE_CONSUMER_KEY and SALESFORCE_CONSUMER_SECRET in .env")

use_cc = os.environ.get("SALESFORCE_USE_USERNAME_PASSWORD", "").lower() not in ("1", "true", "yes")
token_url = os.environ.get("SALESFORCE_TOKEN_URL") or "https://proxi.my.salesforce.com"

try:
    token = get_token_auto(
        ck,
        cs,
        os.environ.get("SALESFORCE_USERNAME") or None,
        os.environ.get("SALESFORCE_PASSWORD") or None,
        use_client_credentials=use_cc,
        security_token=os.environ.get("SALESFORCE_SECURITY_TOKEN") or None,
        use_sandbox=os.environ.get("SALESFORCE_USE_SANDBOX", "").lower() in ("1", "true", "yes"),
        token_url=token_url,
    )
except SalesforceLoginError as e:
    raise RuntimeError(str(e)) from e

INSTANCE_URL = token["instance_url"]
ACCESS_TOKEN = token["access_token"]
print("instance_url:", INSTANCE_URL)
print("access_token length:", len(ACCESS_TOKEN))

instance_url: https://proxi.my.salesforce.com
access_token length: 112


## 6. Build full PATCH body

Uses **`prepare_payload_for_write`** (same as `update_salesforce_job.py`): all mapped `Job__c` fields, picklist coercion, updateable filter. **`HAND_FIELDS`** are merged last.

In [ ]:
from utils.sf_job_payload import prepare_payload_for_write
from utils.sf_job_rest_minimal import describe_sobject

describe = describe_sobject(INSTANCE_URL, ACCESS_TOKEN, JOB_OBJECT)

FIELDS = prepare_payload_for_write(
    JOB_ROW,
    describe,
    use_canonical_description=USE_CANONICAL_DESCRIPTION,
    for_update=True,
    description_use_html=DESCR_USE_HTML,
)

for k, v in HAND_FIELDS.items():
    if v is not None and v != "":
        FIELDS[k] = v

print("Fields in PATCH:", len(FIELDS), "—", sorted(FIELDS.keys()))

Skipped (not updateable on object): Job_Facility_Display__c, Job_Worksite_1_Address__c, Job_Point_of_Contact__c, Job_Standard_Schedule__c, Job_Provider_Start_Date__c, Job_Provider_End_Date__c, Position_Type_DJC__c, Specialty_DJC__c, Occupation_DJC__c, Worksite_Parent__c
Fields in PATCH: 18 — ['External_Job_ID__c', 'External_Job_Link__c', 'Insight__c', 'Job_Account__c', 'Job_City__c', 'Job_Client_Job_Description__c', 'Job_Client_Job_Id__c', 'Job_Dates_Needed__c', 'Job_Patient_Ages__c', 'Job_Ranking__c', 'Job_Recruitment_Level__c', 'Job_State__c', 'Job_Status__c', 'Job_Support_Staff__c', 'Job_Types_of_Cases__c', 'Job_Volume__c', 'Job_Worksite_Location_1__c', 'Salary_Pay_Range__c']


Note: Job_Recruitment_Level__c value 'Critical' not allowed; using first active picklist value 'Building Roster'


## 7. Preview JSON and PATCH

`Job_Client_Job_Description__c` is truncated in the printout only.

In [ ]:
import json

from utils.sf_job_rest_minimal import update_job_record

if not FIELDS:
    raise ValueError("No fields to PATCH — check describe / job row")

_show = dict(FIELDS)
_dk = "Job_Client_Job_Description__c"
if _dk in _show:
    t = str(_show[_dk])
    if len(t) > 1200:
        _show[_dk] = t[:1200] + f"\n… ({len(t)} chars total)"
print(json.dumps(_show, indent=2, default=str))

if DRY_RUN:
    print("\nDRY_RUN: no PATCH. Set DRY_RUN = False in §2 and re-run §5–§7.")
else:
    update_job_record(INSTANCE_URL, ACCESS_TOKEN, JOB_OBJECT, RECORD_ID, FIELDS)
    print(f"\nPATCH sent → {JOB_OBJECT} {RECORD_ID}")

{
  "External_Job_ID__c": "19587",
  "Job_Account__c": "0015f00000HH63kAAD",
  "Job_Worksite_Location_1__c": "0015f00000S30EhAAJ",
  "Job_Client_Job_Id__c": "3159 - Escanaba, MI",
  "Job_Client_Job_Description__c": "<p><strong>General Dentist Locum Tenens Opportunity in Escanaba, MI</strong></p><p><br/></p><p>We are seeking a General Dentist for a locum tenens opportunity in Escanaba, Michigan. This position offers the opportunity to practice comprehensive general dentistry with a supportive clinical team and steady patient flow.</p><p><br/></p><p>This role is ideal for a dentist comfortable with surgical extractions and dentures who enjoys working in a collaborative environment.</p><p><br/></p><p>Travel and lodging may be available for qualified candidates.</p><p><br/></p><p><strong>Dates:</strong> April 7-9<br/><strong>Schedule:</strong> 8a-5p</p><p><br/></p><p><strong>Clinical Scope</strong></p><ul style=\"margin-top:0;margin-bottom:0;padding-left:20px;\"><li style=\"margin:0 0 4px 

RuntimeError: Salesforce REST PATCH HTTP 400: duplicate value found: External_Job_ID__c duplicates value on record with id: a015f00000JPG2lAAH